In [1]:
import face_recognition
import cv2
import pickle
import os

dataset_folder = "data_set"
known_encodings = []
known_names = []

print("ෆොටෝ ටික කියවමින් පවතී. කරුණාකර රැඳී සිටින්න (මෙයට ටික වේලාවක් ගතවනු ඇත)...")

for person_name in os.listdir(dataset_folder):
    person_folder = os.path.join(dataset_folder, person_name)

    if os.path.isdir(person_folder):
        print(f"\n>> {person_name} ගේ ෆොටෝ පරීක්ෂා කරමින්...")

        for image_name in os.listdir(person_folder):
            image_path = os.path.join(person_folder, image_name)

            try:
                # 1. රූපය Load කරගැනීම
                image = face_recognition.load_image_file(image_path)

                # 2. 'cnn' භාවිතා කර මුහුණ තියෙන තැන් (boxes) හොයාගැනීම (GPU එක හරහා)
                boxes = face_recognition.face_locations(image, model="cnn")

                # 3. හොයාගත්ත boxes ඇතුලේ තියෙන මුහුණ num_jitters=10 ක් දාලා හොඳටම Scan කිරීම
                encodings = face_recognition.face_encodings(image, known_face_locations=boxes, num_jitters=10)

                # මුහුණක් හොයාගත්තා නම් විතරක් list එකට එකතු කිරීම
                if len(encodings) > 0:
                    known_encodings.append(encodings[0])
                    known_names.append(person_name)
                    print(f"  ✅ {image_name} - සාර්ථකයි!")
                else:
                    print(f"  ❌ {image_name} - මුහුණක් හොයාගන්න බැරි වුණා!")

            except Exception as e:
                print(f"  ⚠️ {image_name} කියවීමේදී දෝෂයක්: {e}")

# 4. දත්ත ටික Pickle ෆයිල් එකකට Save කිරීම
data = {"encodings": known_encodings, "names": known_names}
with open("face_encodings.pickle", "wb") as f:
    pickle.dump(data, f)

print("\n🎉 වැඩේ ඉවරයි! 'face_encodings.pickle' (AI මොළය) සාර්ථකව හැදුවා!")

ෆොටෝ ටික කියවමින් පවතී. කරුණාකර රැඳී සිටින්න (මෙයට ටික වේලාවක් ගතවනු ඇත)...

>> dulan ගේ ෆොටෝ පරීක්ෂා කරමින්...
  ✅ dulan_17.jpg - සාර්ථකයි!
  ✅ dulan_15.jpg - සාර්ථකයි!
  ✅ dulan_18.jpg - සාර්ථකයි!
  ✅ dulan_19.jpg - සාර්ථකයි!
  ✅ dulan_4.jpg - සාර්ථකයි!
  ✅ dulan_8.jpg - සාර්ථකයි!
  ✅ dulan_11.jpg - සාර්ථකයි!
  ✅ dulan_10.jpg - සාර්ථකයි!
  ✅ dulan_14.jpg - සාර්ථකයි!
  ✅ dulan_12.jpg - සාර්ථකයි!
  ✅ dulan_9.jpg - සාර්ථකයි!
  ✅ dulan_13.jpg - සාර්ථකයි!
  ✅ dulan_16.jpg - සාර්ථකයි!
  ✅ dulan_6.jpg - සාර්ථකයි!

>> isuru ගේ ෆොටෝ පරීක්ෂා කරමින්...
  ✅ isuru_13.jpg - සාර්ථකයි!
  ✅ isuru_8.jpg - සාර්ථකයි!
  ❌ isuru_9.jpg - මුහුණක් හොයාගන්න බැරි වුණා!
  ❌ isuru_16.jpg - මුහුණක් හොයාගන්න බැරි වුණා!
  ❌ isuru_17.jpg - මුහුණක් හොයාගන්න බැරි වුණා!
  ❌ isuru_19.jpg - මුහුණක් හොයාගන්න බැරි වුණා!
  ❌ isuru_11.jpg - මුහුණක් හොයාගන්න බැරි වුණා!
  ✅ isuru_18.jpg - සාර්ථකයි!
  ✅ isuru_5.jpg - සාර්ථකයි!
  ✅ isuru_2.jpg - සාර්ථකයි!
  ✅ isuru_3.jpg - සාර්ථකයි!
  ✅ isuru_7.jpg - සාර්ථකයි!
  ✅ isuru_6.jpg - 

In [2]:

import face_recognition
import cv2
import pickle
import numpy as np
import os

# 1. 'මොළය' (Pickle ෆයිල් එක) Load කරගැනීම
print("AI මොළය Load වෙමින් පවතී...")

# Pickle ෆයිල් එක තියෙනවද කියලා මුලින්ම චෙක් කිරීම
if not os.path.exists("face_encodings.pickle"):
    print("❌ 'face_encodings.pickle' ෆයිල් එක හොයාගන්න බැහැ! කරුණාකර මුලින්ම ඒක හදාගෙන ඉන්න.")
    exit()

with open("face_encodings.pickle", "rb") as f:
    data = pickle.load(f)

known_encodings = data["encodings"]
known_names = data["names"]
print("මොළය සාර්ථකව Load වුණා! ✅")

# 2. කැමරාව පටන් ගැනීම
cap = cv2.VideoCapture(0)

print("කැමරාව ඔන් වුණා! ඔයාව හඳුනාගැනීම ආරම්භ කරමින් පවතී...")

while True:
    ret, frame = cap.read()
    if not ret:
        print("කැමරාවෙන් රූප එන්නේ නෑ!")
        break

    # වේගය සහ නිරවද්‍යතාවය සමතුලිත කරන්න frame එක 50% කින් පොඩි කිරීම (resize)
    small_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)

    # ආලෝකය සමතුලිත කිරීම (CLAHE) - අඳුරු තැන්වලදී වඩාත් පැහැදිලිව මුහුණු හඳුනාගැනීමට
    lab = cv2.cvtColor(small_frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl,a,b))
    enhanced_frame = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # BGR සිට RGB බවට පත් කිරීම
    rgb_small_frame = cv2.cvtColor(enhanced_frame, cv2.COLOR_BGR2RGB)

    # මූණු හොයාගැනීම (Detection) - සජීවී වීඩියෝ සඳහා 'hog' වඩාත් වේගවත්ය
    face_locations = face_recognition.face_locations(rgb_small_frame, model="hog")
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # අපේ අර Pickle එකේ තියෙන මිනුම් එක්ක සැසඳීම (Compare)
        face_distances = face_recognition.face_distance(known_encodings, face_encoding)

        # known_encodings හිස් නැත්නම් පමණක් සැසඳීම කිරීම
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances) # අඩුම දුර තියෙන Index එක හොයනවා

            # 0.65 ට වඩා අඩු නම් විතරයි අඳුරගන්නේ (නිරවද්‍යතාවය වැඩි කර ඇත)
            if face_distances[best_match_index] < 0.65:
                name = known_names[best_match_index]
            else:
                name = "Unknown"
        else:
            name = "Unknown"

        # රූපය 0.5 කින් කුඩා කළ නිසා, අඳින කොටුව නැවත මුල් ප්‍රමාණයට ගේන්න 2 න් ගුණ කිරීම
        top *= 2
        right *= 2
        bottom *= 2
        left *= 2

        # රූපේ උඩ නම සහ කොටුව ඇඳීම
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, name, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)

    # තිරයේ පෙන්වීම
    cv2.imshow('Face Recognition System', frame)

    # 'q' එබුවොත් නතර වීම
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("වැඩේ ඉවරයි! ✨")

AI මොළය Load වෙමින් පවතී...
මොළය සාර්ථකව Load වුණා! ✅
කැමරාව ඔන් වුණා! ඔයාව හඳුනාගැනීම ආරම්භ කරමින් පවතී...


QFontDatabase: Cannot find font directory /home/chamindu-sandeepa/miniconda3/envs/face_env/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/chamindu-sandeepa/miniconda3/envs/face_env/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/chamindu-sandeepa/miniconda3/envs/face_env/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/chamindu-sandeepa/miniconda3/envs/face_env/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for ex

වැඩේ ඉවරයි! ✨
